# Differentiable Optimization with PyTorch

This notebook develops a small differentiable quadratic-program layer from first principles and then trains a neural network through it.

We study the strictly convex unconstrained quadratic program

\[
x^*(Q,p)=\arg\min_x \left(\frac{1}{2}x^\top Qx+p^\top x\right),
\qquad Q\succ 0.
\]

The first-order optimality condition is

\[
Qx^*+p=0,
\]

so the unique optimum is obtained by solving

\[
Qx^*=-p.
\]

In code, we use `torch.linalg.solve` rather than explicitly forming \(Q^{-1}\). PyTorch differentiates through this solve, which lets gradients propagate from a downstream loss through the optimization solution.

This is an educational example of differentiable optimization. It does not implement a custom implicit-differentiation backward pass.

## 1. Imports and reproducibility

The notebook uses only PyTorch and Matplotlib for the core example.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)

## 2. Implement the differentiable QP layer

For each batch element, the layer solves

\[
Qx=-p.
\]

The implementation validates tensor shapes and delegates the numerical linear algebra to `torch.linalg.solve`.

In [ ]:
class UnconstrainedQPLayer(nn.Module):
    """Solve batched strictly convex unconstrained quadratic programs."""

    def forward(self, Q: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
        if Q.ndim < 2 or p.ndim < 1:
            raise ValueError("Q must have shape (..., n, n) and p (..., n).")
        if Q.shape[-1] != Q.shape[-2]:
            raise ValueError("Q must be square in its last two dimensions.")
        if Q.shape[-1] != p.shape[-1]:
            raise ValueError("The last dimension of p must match Q.")
        if Q.shape[:-2] != p.shape[:-1]:
            raise ValueError("Q and p must have matching batch dimensions.")

        return torch.linalg.solve(Q, -p.unsqueeze(-1)).squeeze(-1)

## 3. Solve a small quadratic program

The matrix below is symmetric positive definite, so the objective has a unique global minimizer. We verify the solution using the first-order condition `Q @ x + p = 0`.

In [ ]:
Q = torch.tensor(
    [[2.0, 0.5],
     [0.5, 1.5]],
    dtype=torch.float32,
)

p = torch.tensor([-3.0, -2.0], dtype=torch.float32)

layer = UnconstrainedQPLayer()
x_star = layer(Q.unsqueeze(0), p.unsqueeze(0)).squeeze(0)

residual = Q @ x_star + p

print("Optimal solution:", x_star)
print("Optimality residual:", residual)
print("Residual norm:", torch.linalg.vector_norm(residual).item())

## 4. Verify the gradient

Suppose the downstream scalar loss is

\[
L=\frac{1}{2}\|x^*\|_2^2.
\]

Because

\[
x^*=-Q^{-1}p,
\]

and \(Q\) is symmetric, the analytic gradient with respect to \(p\) is

\[
\frac{\partial L}{\partial p}=-Q^{-1}x^*.
\]

We compare that expression with PyTorch autograd.

In [ ]:
p_grad = p.clone().requires_grad_(True)

x_star_grad = layer(Q.unsqueeze(0), p_grad.unsqueeze(0)).squeeze(0)
loss = 0.5 * torch.sum(x_star_grad ** 2)
loss.backward()

autograd_gradient = p_grad.grad.detach()
analytic_gradient = -torch.linalg.solve(Q, x_star_grad.detach())

print("Autograd gradient:", autograd_gradient)
print("Analytic gradient:", analytic_gradient)
print(
    "Maximum absolute difference:",
    torch.max(torch.abs(autograd_gradient - analytic_gradient)).item(),
)

## 5. Build a predict-then-optimize model

A neural network now predicts the linear objective coefficient \(p\). The predicted \(p\) is passed into the QP layer, and the model output is the optimal decision \(x^*\).

The computational graph is

```text
features
   |
neural network
   |
predicted p
   |
QP solve
   |
optimal decision x*
   |
loss
```

Backpropagation follows this graph in reverse, including through the linear solve.

In [ ]:
class PredictThenOptimize(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        qp_dim: int,
        Q: torch.Tensor,
    ) -> None:
        super().__init__()

        if Q.shape != (qp_dim, qp_dim):
            raise ValueError(f"Q must have shape ({qp_dim}, {qp_dim}).")

        self.predict_p = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, qp_dim),
        )
        self.qp = UnconstrainedQPLayer()
        self.register_buffer("Q", Q.detach().clone())

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        p = self.predict_p(features)
        Q_batch = self.Q.expand(features.shape[0], -1, -1)
        return self.qp(Q_batch, p)

## 6. Create a meaningful synthetic dataset

A tutorial should not train on completely unrelated random inputs and random labels. Instead, we create a known mapping from features to the true QP coefficient `p_true`, then generate targets by solving the corresponding optimization problems.

This gives the model a real signal to learn and makes validation loss interpretable.

In [ ]:
def make_dataset(
    n_samples: int = 600,
    input_dim: int = 6,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    features = torch.randn(n_samples, input_dim)

    Q = torch.tensor(
        [
            [2.5, 0.4, 0.2],
            [0.4, 1.8, 0.3],
            [0.2, 0.3, 1.4],
        ],
        dtype=torch.float32,
    )

    true_weight = torch.tensor(
        [
            [0.8, -0.3, 0.2, 0.5, -0.4, 0.1],
            [-0.2, 0.6, -0.5, 0.1, 0.3, 0.4],
            [0.3, 0.2, 0.7, -0.4, 0.1, -0.6],
        ],
        dtype=torch.float32,
    )
    true_bias = torch.tensor([0.2, -0.1, 0.3], dtype=torch.float32)

    p_true = features @ true_weight.T + true_bias
    p_true = p_true + 0.1 * torch.sin(p_true)

    qp = UnconstrainedQPLayer()
    targets = qp(Q.expand(n_samples, -1, -1), p_true).detach()

    return features, targets, Q


features, targets, Q_train = make_dataset()

split = int(0.8 * len(features))
X_train, X_val = features[:split], features[split:]
y_train, y_val = targets[:split], targets[split:]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

## 7. Train through the optimization layer

We optimize mean squared error between the predicted optimal decision and the ground-truth optimal decision. Both training and validation losses are tracked.

In [ ]:
model = PredictThenOptimize(
    input_dim=X_train.shape[1],
    hidden_dim=32,
    qp_dim=y_train.shape[1],
    Q=Q_train,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
criterion = nn.MSELoss()

train_history = []
val_history = []

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()

    prediction = model(X_train)
    train_loss = criterion(prediction, y_train)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val)

    train_history.append(train_loss.item())
    val_history.append(val_loss.item())

    if epoch == 1 or epoch % 25 == 0:
        print(
            f"Epoch {epoch:3d}/200 | "
            f"train MSE: {train_loss.item():.6f} | "
            f"validation MSE: {val_loss.item():.6f}"
        )

## 8. Inspect learning behavior

A useful result is not merely a falling training loss. Validation loss should also decrease and remain close to training loss.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_history, label="Train")
plt.plot(val_history, label="Validation")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Learning Through a Differentiable QP Layer")
plt.legend()
plt.tight_layout()
plt.show()

print("Final training MSE:", train_history[-1])
print("Final validation MSE:", val_history[-1])

## 9. Confirm that gradients reach the neural network

The optimization layer is useful only if the loss can update parameters before the solver. We explicitly inspect one neural-network parameter gradient.

In [ ]:
model.train()
optimizer.zero_grad()

small_batch_prediction = model(X_train[:32])
small_batch_loss = criterion(small_batch_prediction, y_train[:32])
small_batch_loss.backward()

first_weight = model.predict_p[0].weight
gradient_norm = torch.linalg.vector_norm(first_weight.grad).item()

print("Gradient norm of the first neural-network layer:", gradient_norm)

## 10. Optional: constructing a learnable positive-definite Q

If a model predicts the quadratic matrix itself, an arbitrary matrix cannot be used as \(Q\). One simple parameterization is

\[
Q=LL^\top+\varepsilon I,
\]

which guarantees strict positive definiteness for \(\varepsilon>0\).

In [ ]:
def positive_definite_from_factor(
    factor: torch.Tensor,
    jitter: float = 1e-3,
) -> torch.Tensor:
    if factor.ndim < 2 or factor.shape[-1] != factor.shape[-2]:
        raise ValueError("factor must have shape (..., n, n).")
    if jitter <= 0:
        raise ValueError("jitter must be strictly positive.")

    n = factor.shape[-1]
    identity = torch.eye(n, dtype=factor.dtype, device=factor.device)
    return factor @ factor.transpose(-1, -2) + jitter * identity


factor = torch.randn(3, 3)
Q_learnable = positive_definite_from_factor(factor)
eigenvalues = torch.linalg.eigvalsh(Q_learnable)

print("Eigenvalues:", eigenvalues)
print("All eigenvalues are positive:", bool(torch.all(eigenvalues > 0)))

## 11. Where constrained problems differ

For a constrained quadratic program,

\[
\begin{aligned}
\min_x \quad & \frac{1}{2}x^\top Qx+p^\top x \\
\text{s.t.}\quad & Gx\le h,
\end{aligned}
\]

the solution is no longer given by a single unconstrained linear solve. General constrained problems should be handled by a proper optimization solver.

This repository includes a separate CVXPYLayers example in `examples/constrained_qp_cvxpylayers.py`. That example is intentionally separate from this PyTorch-only tutorial so that the numerical method and differentiation mechanism are not conflated.

## Takeaways

1. An optimization solution can be part of a PyTorch computational graph.
2. `torch.linalg.solve` is differentiable, so an exact unconstrained QP solve can be embedded in a neural network.
3. The optimization problem must be mathematically well posed; here, \(Q\) is symmetric positive definite.
4. Synthetic demonstrations should contain a learnable relationship between inputs and targets.
5. Training loss alone is insufficient; validation behavior and gradient checks provide stronger evidence.
6. Constrained differentiable optimization generally requires a solver designed for that problem class.

The implementation here is deliberately small enough to explain line by line while remaining mathematically accurate.